In [ ]:
import os
import sys
import glob
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.signal import welch

# Define the root of the repository
REPO_ROOT = "/home/iec/MinhHieu/rPPG"
if not os.path.exists(REPO_ROOT):
    REPO_ROOT = r"d:\New folder\Non-Invasive\rPPG"

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.PhysFormer import ViT_ST_ST_Compact3_TDC_gra_sharp
from neural_methods.loss.PhysNetNegPearsonLoss import Neg_Pearson
from neural_methods.loss.PhysFormerLossComputer import TorchLossComputer


In [ ]:
# ----- params -----
PREPROCESSED_PATH = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupD")
OUTPUT_MODEL_DIR  = os.path.join(REPO_ROOT, "final_model_release")
CHUNK_LENGTH = 160
IMG_H, IMG_W = 128, 128
VIDEO_FPS = 30
EPOCHS = 30
BATCH_SIZE = 4
LR = 0.0001

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)


In [ ]:
# PyTorch Dataset + DataLoader
class PhysFormerDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [f.replace("input", "label") for f in self.inputs]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (T,)

        # NDHWC -> NCDHW: transpose (3, 0, 1, 2)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]
        chunk_id   = fname[split_idx + 6:].split(".")[0]

        return data, label, subject_id, chunk_id

all_input_files = glob.glob(os.path.join(PREPROCESSED_PATH, "*", "*_input*.npy"))
dataset = PhysFormerDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
print(f"DataLoader ready: {len(train_loader)} batches")


In [ ]:
# HR calculation based on ground truth label
def get_hr(y, sr=30, min_hr=30, max_hr=180):
    p, q = welch(y, sr, nfft=1e5/sr, nperseg=np.min((len(y)-1, 256)))
    return p[(p>min_hr/60)&(p<max_hr/60)][np.argmax(q[(p>min_hr/60)&(p<max_hr/60)])]*60

# Model
model = ViT_ST_ST_Compact3_TDC_gra_sharp(
    image_size=(CHUNK_LENGTH, IMG_H, IMG_W),
    patches=(4, 4, 4),
    dim=96,
    ff_dim=144,
    num_heads=4,
    num_layers=12,
    dropout_rate=0.2,
    theta=0.7
)
model = model.to(DEVICE)

criterion_Pearson = Neg_Pearson()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=0.00005)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)


In [ ]:
# Training loop
a_start = 1.0
b_start = 1.0
exp_b = 1.0

for epoch in range(EPOCHS):
    print(f"\n====Training Epoch: {epoch}====")
    loss_rPPG_avg = []
    loss_peak_avg = []
    loss_kl_avg_test = []
    loss_hr_mae = []
    
    model.train()
    tbar = tqdm(train_loader, ncols=80)
    for idx, batch in enumerate(tbar):
        data, label = batch[0], batch[1]
        
        # Calculate ground truth HR for frequency loss
        hr = torch.tensor([get_hr(i.numpy(), sr=VIDEO_FPS) for i in label]).float().to(DEVICE)
        
        data = data.float().to(DEVICE)
        label = label.float().to(DEVICE)
        
        optimizer.zero_grad()
        
        gra_sharp = 2.0
        rPPG, _, _, _ = model(data, gra_sharp)
        rPPG = (rPPG - torch.mean(rPPG, axis=-1).view(-1, 1)) / torch.std(rPPG, axis=-1).view(-1, 1)
        
        loss_rPPG = criterion_Pearson(rPPG, label)
        
        fre_loss = 0.0
        kl_loss = 0.0
        train_mae = 0.0
        
        for bb in range(data.shape[0]):
            loss_distribution_kl, fre_loss_temp, train_mae_temp = TorchLossComputer.cross_entropy_power_spectrum_DLDL_softmax2(
                rPPG[bb],
                hr[bb],
                VIDEO_FPS,
                std=1.0
            )
            fre_loss += fre_loss_temp
            kl_loss += loss_distribution_kl
            train_mae += train_mae_temp
            
        fre_loss /= data.shape[0]
        kl_loss /= data.shape[0]
        train_mae /= data.shape[0]
        
        if epoch > 10:
            a = 0.05
            b = 5.0
        else:
            a = a_start
            b = b_start * math.pow(exp_b, epoch / 10.0)
            
        loss = a * loss_rPPG + b * (fre_loss + kl_loss)
        loss.backward()
        optimizer.step()
        
        loss_rPPG_avg.append(float(loss_rPPG.data))
        loss_peak_avg.append(float(fre_loss.data))
        loss_kl_avg_test.append(float(kl_loss.data))
        loss_hr_mae.append(float(train_mae))
        
        if idx % 10 == 9:
            tbar.set_description(f"Loss: {loss.item():.4f}")
            
    scheduler.step()
    print(f"Epoch {epoch} summary: NegPearson={np.mean(loss_rPPG_avg):.4f}, KL={np.mean(loss_kl_avg_test):.4f}, CE={np.mean(loss_peak_avg):.4f}, MAE={np.mean(loss_hr_mae):.4f}")

print("Training Complete!")


In [ ]:
# Save final model
output_path = os.path.join(OUTPUT_MODEL_DIR, "GroupD_PhysFormer.pth")
torch.save(model.state_dict(), output_path)
print(f"Weights saved to {output_path}")
